## Data Engineering Project

This project demonstrates an ETL pipeline where raw healthcare data is extracted from CSVs, transformed using SQL to ensure data integrity and business and clinical accuracy, and loaded for downstream analytics.

The dataset was obtained from https://www.kaggle.com/datasets/rajkumarpadmanabhan/ca-hospital-dataset-q1-2025/data

The dataset contains 9 tables: patients, encounters, diagnoses, procedures, medications, lab_tests, claims_and_billing, providers, and denials.

This synthetic dataset simulates the end-to-end operations of a California-based hospital for Q1 2025. It includes over 126,000 rows across 9 fully integrated tables that capture patient visits, clinical procedures, diagnoses, lab tests, medication prescriptions, provider details, billing, claims, and denials — designed for data analytics, machine learning, and healthcare research.

Tables Included: patients.csv – Patient demographics, insurance, DOB, gender

encounters.csv – Admission/discharge details, visit types, departments

diagnoses.csv – ICD-10 diagnosis codes linked to encounters

procedures.csv – CPT/ICD-10-PCS procedure codes per patient

medications.csv – Drug names, dosages, prescription data

lab_tests.csv – Test names, result values, normal ranges

claims_and_billing.csv – Financial charges, insurance claims, payments

providers.csv – Doctors, specializations, provider roles

denials.csv – Reasons for claim denial, status, appeal info

This dataset is completely synthetic and safe for public use. It was generated using custom rules, distributions, and logic reflective of real hospital operations.

### 1. INGESTION

I used the read_files function because it is the most modern and reliable way to load data in Databricks. Since the datasets live in Unity Catalog Volumes, it instantly recognises the data stored there and automatically identify the correct column data types (like dates and numbers) directly from the CSV files. Also, using DROP TABLE IF EXISTS supports idempotency where anyone can run your notebook from top to bottom multiple times without it breaking.

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS bronze_hospital;
USE bronze_hospital;

In [0]:
%sql

-- Patients Ingestion
DROP TABLE IF EXISTS patients;
CREATE TABLE patients AS
SELECT * FROM read_files(
  "/Volumes/workspace/default/hospital_dataset/patients.csv",
  format => "csv",
  header => true,
  inferSchema => true);

-- Encounters Ingestion
DROP TABLE IF EXISTS encounters;
CREATE TABLE encounters AS
SELECT * FROM read_files(
  "/Volumes/workspace/default/hospital_dataset/encounters.csv",
  format => "csv",
  header => true,
  inferSchema => true);

-- Diagnoses Ingestion
DROP TABLE IF EXISTS diagnoses;
CREATE TABLE diagnoses AS 
SELECT * FROM read_files(
  "/Volumes/workspace/default/hospital_dataset/diagnoses.csv", 
  format => "csv", 
  header => true, 
  inferSchema => true);

-- Procedures Ingestion
DROP TABLE IF EXISTS procedures;
CREATE TABLE procedures AS 
SELECT * FROM read_files(
  "/Volumes/workspace/default/hospital_dataset/procedures.csv", 
  format => "csv", 
  header => true, 
  inferSchema => true);

-- Medications Ingestion
DROP TABLE IF EXISTS medications;
CREATE TABLE medications AS 
SELECT * FROM read_files(
  "/Volumes/workspace/default/hospital_dataset/medications.csv", 
  format => "csv", 
  header => true, 
  inferSchema => true);

-- Laboratory Tests Ingestion
DROP TABLE IF EXISTS lab_tests;
CREATE TABLE lab_tests AS 
SELECT * FROM read_files(
  "/Volumes/workspace/default/hospital_dataset/lab_tests.csv", 
  format => "csv", 
  header => true, 
  inferSchema => true);

-- Claims and Billings Ingestion
DROP TABLE IF EXISTS claims_and_billing;
CREATE TABLE claims_and_billing AS 
SELECT * FROM read_files(
  "/Volumes/workspace/default/hospital_dataset/claims_and_billing.csv", 
  format => "csv", 
  header => true, 
  inferSchema => true);

-- Providers Ingestion
DROP TABLE IF EXISTS providers;
CREATE TABLE providers AS 
SELECT * FROM read_files(
  "/Volumes/workspace/default/hospital_dataset/providers.csv", 
  format => "csv", 
  header => true, 
  inferSchema => true);

-- Denials Ingestion
DROP TABLE IF EXISTS denials;
CREATE TABLE denials AS 
SELECT * FROM read_files(
  "/Volumes/workspace/default/hospital_dataset/denials.csv", 
  format => "csv", 
  header => true, 
  inferSchema => true);


In [0]:
%sql
SHOW TABLES IN bronze_hospital;

### 2. TRANSFORM

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS silver_hospital;


2.1. Cleaning Patients Table

In [0]:
%sql

-- Check data quality
SELECT * FROM bronze_hospital.patients
LIMIT 100;


Observations and considerations:
1. patient_id, a key_identifier starts with PAT and six numbers
2. Ensure first and last name formats are consistent (First, Last)
3. DOB and dates to be consistent as dates
4. For categorical entries (eg. Gender, Ethnicity, etc), to ensure categories reflect the real data gap, we will treat 'Unknown' catgeory as NULL.
5. Ensure emails are consistent in format (lowercase)
6. Consider calculating age using current date and DOB for later analysis
7. Remove rows with null patient_id
8. Phone numbers are not consistent, some have +1, others (), then some with . in between numbers, will remove non-numeric char


AUDIT CHECK:
Check for NULLs. Some columns have 'Unknown' entries so will convert this to NULL.

In [0]:
%sql

-- '1' if the data is GOOD, and '0' if it is BAD (NULL or 'Unknown')
WITH patients_quality_flags AS (
  SELECT
    -- Ids
    CASE WHEN patient_id IS NULL THEN 1 ELSE 0 END AS null_id,
  
    -- Demographics
    CASE WHEN first_name IS NULL THEN 1 ELSE 0 END AS null_first_name,
    CASE WHEN last_name IS NULL THEN 1 ELSE 0 END AS null_last_name,
    CASE WHEN dob IS NULL THEN 1 ELSE 0 END AS null_dob,
    CASE WHEN gender IS NULL OR gender = 'Unknown' THEN 1 ELSE 0 END AS null_gender,
    CASE WHEN ethnicity IS NULL OR ethnicity = 'Unknown' THEN 1 ELSE 0 END AS null_ethnicity,
  
    -- Contact info
    CASE WHEN phone IS NULL THEN 1 ELSE 0 END AS null_phone,
    CASE WHEN email IS NULL THEN 1 ELSE 0 END AS null_email,
  
    -- Location
    CASE WHEN address IS NULL THEN 1 ELSE 0 END AS null_address,
    CASE WHEN city IS NULL THEN 1 ELSE 0 END AS null_city,
    CASE WHEN state IS NULL THEN 1 ELSE 0 END AS null_state,
    CASE WHEN zip IS NULL THEN 1 ELSE 0 END AS null_zip,
  
    -- Social/Insurance
    CASE WHEN marital_status IS NULL OR marital_status = 'Unknown' THEN 1 ELSE 0 END AS null_marital,
    CASE WHEN insurance_type IS NULL OR insurance_type = 'Unknown' THEN 1 ELSE 0 END AS null_insurance,
    CASE WHEN registration_date IS NULL THEN 1 ELSE 0 END AS null_registration_date

FROM bronze_hospital.patients
)

-- Calculate percentages of nulls in each column
SELECT
  COUNT(*) AS total_rows,
  ROUND(AVG(null_id) * 100, 2) AS pct_null_id,
  ROUND(AVG(null_first_name) * 100, 2) AS pct_null_first_name,
  ROUND(AVG(null_last_name) * 100, 2) AS pct_null_last_name,
  ROUND(AVG(null_dob) * 100, 2) AS pct_null_dob,
  ROUND(AVG(null_gender) * 100, 2) AS pct_null_gender,
  ROUND(AVG(null_ethnicity) * 100, 2) AS pct_null_ethnicity,
  ROUND(AVG(null_phone) * 100, 2) AS pct_null_phone,
  ROUND(AVG(null_email) * 100, 2) AS pct_null_email,
  ROUND(AVG(null_address) * 100, 2) AS pct_null_address,
  ROUND(AVG(null_city) * 100, 2) AS pct_null_city,
  ROUND(AVG(null_state) * 100, 2) AS pct_null_state,
  ROUND(AVG(null_zip) * 100, 2) AS pct_null_zip,
  ROUND(AVG(null_marital) * 100, 2) AS pct_null_marital,
  ROUND(AVG(null_insurance) * 100, 2) AS pct_null_insurance,
  ROUND(AVG(null_registration_date) * 100, 2) AS pct_null_registration_date
FROM patients_quality_flags;
    

In [0]:
%sql

-- Check duplicates
SELECT COUNT(*)
FROM bronze_hospital.patients
GROUP BY patient_id
HAVING COUNT(*) > 1;



- There were 60,000 unique instances/rows.
- Phone, Email, Address, City, State, Zip, Marital Status are columns with NULLS.
- Only Email has NULLs accounting to ~ 20%, the rest have NULLs less than 10%.

In [0]:
%sql

-- Cleaning Patients Table
CREATE OR REPLACE TABLE silver_hospital.patients AS
SELECT
  TRIM(patient_id) as patient_id,
  INITCAP(TRIM(first_name)) AS first_name,
  INITCAP(TRIM(last_name)) AS last_name,

  -- TRY_CAST for safer date conversions
  TRY_CAST(dob AS DATE) AS birth_date,
  
  -- Age calculation: using TRY_CAST ensures 'Unknown' dates don't break the math
  FLOOR(DATEDIFF(CURRENT_DATE(), TRY_CAST(dob AS DATE)) / 365.25) AS age,
  
  NULLIF(gender, 'Unknown') AS gender,
  NULLIF(ethnicity, 'Unknown') AS ethnicity,
  NULLIF(insurance_type, 'Unknown') AS insurance_type,
  NULLIF(marital_status, 'Unknown') AS marital_status,
  NULLIF(address, 'Unknown') AS address,
  NULLIF(city, 'Unknown') AS city,
  NULLIF(state, 'Unknown') AS state,
  
  -- zip encountered BIGINT errors; handle as STRING first
  NULLIF(TRIM(CAST(zip AS STRING)), 'Unknown') AS zip, 
  
  -- Clean phone numbers and ensure they remain strings
  regexp_replace(CAST(phone AS STRING), '[^0-9]', '') AS phone, 
  
  NULLIF(email, 'Unknown') AS email,
  TRY_CAST(registration_date AS DATE) as registration_date

FROM bronze_hospital.patients

-- Filter out records with no ID
WHERE patient_id IS NOT NULL 
  AND patient_id != 'Unknown';

-- Check data quality
SELECT * FROM silver_hospital.patients
LIMIT 100;



2.2. Cleaning Encounters Table

In [0]:
%sql

-- Check table
SELECT *
FROM bronze_hospital.encounters
LIMIT 100;


Observations:
- encounter_id, patient_id and provider_id has consistent format (3 LETTERS followed by numbers)
- there are dates in this table, we will ensure it is in date format
- there are no unusual formatting errors that stood out
- lots of NULLs with admission type, discharge date, length of stay
- numeric columns are on their correct data type

In [0]:
%sql

WITH encounters_quality_flags AS (
SELECT
  CASE WHEN encounter_id IS NULL THEN 1 ELSE 0 END AS null_encounter_id,
  CASE WHEN patient_id IS NULL THEN 1 ELSE 0 END AS null_patient_id,
  CASE WHEN provider_id IS NULL THEN 1 ELSE 0 END AS null_provider_id,
  CASE WHEN visit_date IS NULL THEN 1 ELSE 0 END AS null_visit_date,
  CASE WHEN visit_type IS NULL THEN 1 ELSE 0 END AS null_visit_type,
  CASE WHEN department IS NULL THEN 1 ELSE 0 END AS null_department,
  CASE WHEN reason_for_visit IS NULL THEN 1 ELSE 0 END AS null_reason_for_visit,
  CASE WHEN diagnosis_code IS NULL THEN 1 ELSE 0 END AS null_diagnosis_code,
  CASE WHEN admission_type IS NULL THEN 1 ELSE 0 END AS null_admission_type,
  CASE WHEN discharge_date IS NULL THEN 1 ELSE 0 END AS null_discharge_date,
  CASE WHEN length_of_stay IS NULL THEN 1 ELSE 0 END AS null_length_of_stay,
  CASE WHEN status IS NULL THEN 1 ELSE 0 END AS null_status,
  CASE WHEN readmitted_flag IS NULL THEN 1 ELSE 0 END AS null_readmitted_flag
FROM bronze_hospital.encounters
)

SELECT
  COUNT(*) AS total_rows,
  ROUND(AVG(null_encounter_id) * 100, 2) AS pct_null_encounter_id,
  ROUND(AVG(null_patient_id) * 100, 2) AS pct_null_patient_id,
  ROUND(AVG(null_provider_id) * 100, 2) AS pct_null_provider_id,
  ROUND(AVG(null_visit_date) * 100, 2) AS pct_null_visit_date,
  ROUND(AVG(null_visit_type) * 100, 2) AS pct_null_visit_type,
  ROUND(AVG(null_department) * 100, 2) AS pct_null_department,
  ROUND(AVG(null_reason_for_visit) * 100, 2) AS pct_null_reason_for_visit,
  ROUND(AVG(null_diagnosis_code) * 100, 2) AS pct_null_diagnosis_code,
  ROUND(AVG(null_admission_type) * 100, 2) AS pct_null_admission_type,
  ROUND(AVG(null_discharge_date) * 100, 2) AS pct_null_discharge_date,
  ROUND(AVG(null_length_of_stay) * 100, 2) AS pct_null_length_of_stay,
  ROUND(AVG(null_status) * 100, 2) AS pct_null_status,
  ROUND(AVG(null_readmitted_flag) * 100, 2) AS pct_null_readmitted_flag
FROM encounters_quality_flags;


In [0]:
%sql

-- Check for duplicates
SELECT COUNT(*)
FROM bronze_hospital.encounters
GROUP BY patient_id, encounter_id
HAVING COUNT(*) > 1;


- There were 70,000 rows
- Admission type and discharge date columns have more than 60% missing data

In [0]:
%sql

-- Cleaning Encounters Table

CREATE OR REPLACE TABLE silver_hospital.encounters AS
SELECT
  encounter_id,
  patient_id,
  provider_id,
  TRY_CAST(visit_date AS DATE) AS visit_date,
  visit_type,
  department,
  reason_for_visit,
  diagnosis_code,
  admission_type,
  TRY_CAST(discharge_date AS DATE) as discharge_date,
  TRY_CAST(length_of_stay AS INT) as length_of_stay,
  status,
  readmitted_flag
FROM bronze_hospital.encounters
WHERE patient_id IS NOT NULL
  AND encounter_id IS NOT NULL
  AND provider_id IS NOT NULL;

-- Check data generated
SELECT * FROM silver_hospital.encounters
LIMIT 100;


I used TRY_CAST here to identify malformed string it can't convert and simply returns it as NULL.